In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


def find_data_dir():
    target = Path("Data/GEFCom2017FinalMatch_4level")
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / target
        if candidate.exists():
            return candidate
    return Path.cwd()

DATA_DIR = find_data_dir()
LOAD_FILLED_PATH = DATA_DIR / "load_final_filled.csv"
HIER_PATH = DATA_DIR / "hierarchy.csv"
OUTPUT_PATH = DATA_DIR / "load_final_filled.csv"  # we do not rewrite this file
SUM_MATRIX_PATH = DATA_DIR / "sum_matrix.csv"
HIER_INFO_PATH = DATA_DIR / "hierarchy_info.json"
NODE_VALUES_PATH = DATA_DIR / "node_values.npy"
NORM_PATH = DATA_DIR / "normalization_params.npy"
NORM_CSV_PATH = DATA_DIR / "node_values_normalized.csv"

LOG_OFFSET = 1.0
LOG_SKEW_THRESHOLD = 1.0
LOG_RATIO_THRESHOLD = 10.0
NORM_SKEW_THRESHOLD = 1.0
NORM_KURTOSIS_THRESHOLD = 5.0
TRAIN_RATIO = 0.8  # fit normalization stats on first 80% to avoid test leakage

print(f"Data directory: {DATA_DIR}")
print(f"Input filled file: {LOAD_FILLED_PATH}")
print(f"Hierarchy file: {HIER_PATH}")


In [ ]:
# Step 1: Read load_final_filled.csv as hierarchical data
df_nodes = pd.read_csv(LOAD_FILLED_PATH, index_col=0)
df_nodes.index = pd.to_datetime(df_nodes.index)
df_nodes = df_nodes.apply(pd.to_numeric, errors="coerce")
print(f"Input data shape: {df_nodes.shape}")
print(f"Columns sample: {list(df_nodes.columns)[:10]} ... total {len(df_nodes.columns)}")
print(f"Index range: {df_nodes.index.min()} to {df_nodes.index.max()}")


In [ ]:
# Step 2: Parse hierarchy and enforce column order (Top -> Middle1 -> Middle2 -> Bottom)
import csv
import json
from functools import lru_cache


def normalize_node(value):
    if value is None:
        return None
    text = str(value).strip()
    if text == "":
        return None
    return text

children = {}
top_order = []
mid1_order = []
mid2_order = []
bottom_order = []

with HIER_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        top = normalize_node(row.get("Top"))
        mid1 = normalize_node(row.get("Middle1"))
        mid2 = normalize_node(row.get("Middle2"))
        bottom = normalize_node(row.get("Bottom"))
        path = [p for p in (top, mid1, mid2, bottom) if p is not None]
        if not path:
            continue
        if top is not None and top not in top_order:
            top_order.append(top)
        if mid1 is not None and mid1 not in mid1_order:
            mid1_order.append(mid1)
        if mid2 is not None and mid2 not in mid2_order:
            mid2_order.append(mid2)
        if bottom is not None and bottom not in bottom_order:
            bottom_order.append(bottom)
        for parent, child in zip(path[:-1], path[1:]):
            children.setdefault(parent, [])
            if child not in children[parent]:
                children[parent].append(child)

mid_order = mid1_order + mid2_order
node_order = top_order + mid_order + bottom_order

print("Hierarchy structure:")
print(f"Top: {top_order}")
print(f"Middle1: {mid1_order}")
print(f"Middle2: {mid2_order}")
print(f"Bottom (sample): {bottom_order[:10]} ... total {len(bottom_order)}")

# Ensure df_nodes columns follow hierarchy order
missing_in_data = [c for c in node_order if c not in df_nodes.columns]
if missing_in_data:
    print(f"Warning: nodes in hierarchy but missing in data: {missing_in_data[:10]} ... total {len(missing_in_data)}")
present_order = [c for c in node_order if c in df_nodes.columns]
df_nodes = df_nodes.loc[:, present_order]
print(f"Reordered data shape: {df_nodes.shape}")

In [ ]:
# Step 3: Build sum matrix and hierarchy info (metadata only; does not modify input CSVs)
bottom_idx = {n: i for i, n in enumerate(bottom_order)}
index_map = {n: i for i, n in enumerate(node_order)}

@lru_cache(None)
def get_children(node):
    return children.get(node, [])

@lru_cache(None)
def bottoms(node):
    if node in bottom_idx:
        return [node]
    res = []
    for ch in get_children(node):
        for b in bottoms(ch):
            if b not in res:
                res.append(b)
    return res

matrix = [[0] * len(bottom_order) for _ in node_order]
for i, node in enumerate(node_order):
    for b in bottoms(node):
        matrix[i][bottom_idx[b]] = 1

with SUM_MATRIX_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(matrix)

print(f"Sum matrix shape: ({len(node_order)}, {len(bottom_order)})")
print("Sample sum matrix rows:")
for i, row in enumerate(matrix[:3]):
    print(f"  Row {i}: {row}")

# Create hierarchy info
mid_to_bottom_indices = []
for mid in mid_order:
    idxs = []
    for b in bottoms(mid):
        idxs.append(index_map[b])
    mid_to_bottom_indices.append(idxs)

hierarchy_info = {
    "num_total_nodes": len(node_order),
    "num_bottom_nodes": len(bottom_order),
    "bottom_start_idx": len(top_order) + len(mid_order),
    "num_mid_nodes": len(mid_order),
    "num_top_nodes": len(top_order),
    "excluded_nodes": ["E004", "23", "28"],
    "excluded_nodes_note": "Excluded from the modeling hierarchy and canonical load_final_filled.csv; node_order, sum_matrix.csv, and node_values.npy use the filtered 158-node hierarchy.",
    "middle_levels": [
        list(range(len(top_order), len(top_order) + len(mid1_order))),
        list(range(len(top_order) + len(mid1_order), len(top_order) + len(mid_order))),
    ],
    "middle_levels_provenance": {
        "generated_by": "DataProcessing.ipynb",
        "source": "hierarchy.csv",
        "validated_against": ["sum_matrix.csv", "mid_to_bottom_indices"],
    },
    "top_nodes": top_order,
    "mid_nodes": mid_order,
    "bottom_nodes": bottom_order,
    "node_order": node_order,
    "mid_to_bottom_indices": mid_to_bottom_indices,
}

with HIER_INFO_PATH.open("w", encoding="utf-8") as f:
    json.dump(hierarchy_info, f, ensure_ascii=True, indent=2)

print("Hierarchy info:")
print(f"  Total nodes: {hierarchy_info['num_total_nodes']}")
print(f"  Top nodes ({len(top_order)}): {top_order}")
print(f"  Middle nodes ({len(mid_order)}): {mid_order[:10]} ...")
print(f"  Bottom nodes ({len(bottom_order)}): {bottom_order[:10]} ...")
print("Files saved:")
print(f"  Sum matrix: {SUM_MATRIX_PATH}")
print(f"  Hierarchy info: {HIER_INFO_PATH}")


In [ ]:
# Step 4: Normalization and prepare final values
def log_transform(df, log_offset=1.0):
    return np.log(df + log_offset)


def minmax_normalize(df):
    data = df.values
    min_val = float(data.min())
    max_val = float(data.max())
    denom = max_val - min_val if max_val != min_val else 1.0
    data_norm = (df - min_val) / denom
    params = {"method": "minmax", "min": min_val, "max": max_val}
    return data_norm, params


def zscore_normalize(df):
    data = df.values
    mean_val = float(data.mean())
    std_val = float(data.std())
    denom = std_val if std_val != 0 else 1.0
    data_norm = (df - mean_val) / denom
    params = {"method": "zscore", "mean": mean_val, "std": std_val}
    return data_norm, params


def should_log_transform(series, skew_threshold=1.0, ratio_threshold=10.0):
    values = series.to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return False
    if values.min() < 0:
        return False
    positive = values[values > 0]
    if positive.size == 0:
        return False
    ratio = values.max() / positive.min()
    skew = pd.Series(values).skew()
    if skew is not None and skew > skew_threshold:
        return True
    return ratio > ratio_threshold


def choose_norm_method(series, skew_threshold=1.0, kurtosis_threshold=5.0):
    values = series.to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return "minmax", {"skew": None, "kurtosis": None}
    stats = pd.Series(values)
    skew = float(stats.skew())
    kurtosis = float(stats.kurtosis())
    if abs(skew) <= skew_threshold and abs(kurtosis) <= kurtosis_threshold:
        return "zscore", {"skew": skew, "kurtosis": kurtosis}
    return "minmax", {"skew": skew, "kurtosis": kurtosis}


def normalize_dataframe(df, log_offset=1.0, force_log=None, force_norm=None, train_ratio=TRAIN_RATIO):
    """Fit normalization on the first ``train_ratio`` fraction of rows only,
    then apply to the whole DataFrame.

    This prevents train/test leakage: the test period does not influence the
    log-transform decision, the norm-method decision, nor the fitted
    (mean/std) or (min/max) parameters.
    """
    T = len(df)
    train_T = max(1, int(T * train_ratio))
    df_train = df.iloc[:train_T]

    use_log = force_log if force_log is not None else should_log_transform(
        df_train.stack(),
        skew_threshold=LOG_SKEW_THRESHOLD,
        ratio_threshold=LOG_RATIO_THRESHOLD,
    )

    if use_log:
        df_base = log_transform(df, log_offset=log_offset)
        df_base_train = df_base.iloc[:train_T]
        data_space = "log"
    else:
        df_base = df.copy()
        df_base_train = df_train.copy()
        data_space = "raw"

    norm_method, stats = choose_norm_method(
        df_base_train.stack(),
        skew_threshold=NORM_SKEW_THRESHOLD,
        kurtosis_threshold=NORM_KURTOSIS_THRESHOLD,
    )
    if force_norm is not None:
        norm_method = force_norm

    # Fit on TRAIN, apply to FULL series
    if norm_method == "zscore":
        mean_val = float(df_base_train.values.mean())
        std_val = float(df_base_train.values.std())
        denom = std_val if std_val != 0 else 1.0
        data_norm = (df_base - mean_val) / denom
        norm_params = {"method": "zscore", "mean": mean_val, "std": std_val}
    else:
        min_val = float(df_base_train.values.min())
        max_val = float(df_base_train.values.max())
        denom = max_val - min_val if max_val != min_val else 1.0
        data_norm = (df_base - min_val) / denom
        norm_params = {"method": "minmax", "min": min_val, "max": max_val}

    params = {
        "use_log": bool(use_log),
        "log_offset": float(log_offset) if use_log else None,
        "data_space": data_space,
        "norm_method": norm_method,
        "decision_stats": stats,
        "train_ratio": float(train_ratio),
        "train_T": int(train_T),
        "total_T": int(T),
    }
    params.update(norm_params)

    values = data_norm.to_numpy(dtype=np.float32).reshape(-1, df.shape[1], 1)
    return values, params, use_log, norm_method
print(f"Input data for normalization shape: {df_nodes.shape}")
print(f"Columns (ordered): {list(df_nodes.columns)[:10]} ... total {len(df_nodes.columns)}")

# Normalize
values, norm_params, use_log, norm_method = normalize_dataframe(df_nodes, log_offset=LOG_OFFSET)

# Save normalized values and parameters
np.save(NODE_VALUES_PATH, values)
np.save(NORM_PATH, norm_params)
df_norm = pd.DataFrame(values.squeeze(-1), index=df_nodes.index, columns=df_nodes.columns)
df_norm.to_csv(NORM_CSV_PATH)
print(f"Saved normalized csv to {NORM_CSV_PATH}")

print("Normalization completed:")
print(f"  Use log transform: {use_log}")
print(f"  Normalization method: {norm_method}")
print(f"  Normalized values shape: {values.shape}")
print(f"  Normalization parameters: {norm_params}")
print("Files saved:")
print(f"  Node values: {NODE_VALUES_PATH}")
print(f"  Normalization params: {NORM_PATH}")
